## Pemodelan & Training – XGBoost (Binary Classification)
---
Input  : `X_train.csv`, `y_train.csv`, `X_test.csv`, `y_test.csv`  
Output : `xgboost_binary_model.pkl` + laporan evaluasi lengkap  
Target : **1 = Kecanduan**, **0 = Tidak Kecanduan**

### 3.1 Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
from xgboost import XGBClassifier

from sklearn.model_selection import (
    StratifiedKFold, cross_val_score, RandomizedSearchCV
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay, average_precision_score
)

print(f'✅ XGBoost version : {xgb.__version__}')
print('✅ Semua library berhasil diimport')

### 3.2 Load Data

In [ ]:
X_train = pd.read_csv(r'E:\Python - Project\phone-addiction-detection\outputs\Dataset-user-behavior\FeatureEng-out\X_train.csv')
X_test  = pd.read_csv(r'E:\Python - Project\phone-addiction-detection\outputs\Dataset-user-behavior\FeatureEng-out\X_test.csv')
y_train = pd.read_csv(r'E:\Python - Project\phone-addiction-detection\outputs\Dataset-user-behavior\FeatureEng-out\y_train.csv').squeeze()
y_test  = pd.read_csv(r'E:\Python - Project\phone-addiction-detection\outputs\Dataset-user-behavior\FeatureEng-out\y_test.csv').squeeze()

print(f'X_train shape : {X_train.shape}')
print(f'X_test  shape : {X_test.shape}')
print(f'\nDistribusi y_train:')
print(f'  0 (Tidak Kecanduan): {(y_train==0).sum()} ({(y_train==0).sum()/len(y_train)*100:.1f}%)')
print(f'  1 (Kecanduan)      : {(y_train==1).sum()} ({(y_train==1).sum()/len(y_train)*100:.1f}%)')
print(f'\nDistribusi y_test:')
print(f'  0 (Tidak Kecanduan): {(y_test==0).sum()} ({(y_test==0).sum()/len(y_test)*100:.1f}%)')
print(f'  1 (Kecanduan)      : {(y_test==1).sum()} ({(y_test==1).sum()/len(y_test)*100:.1f}%)')

# Hitung scale_pos_weight untuk menangani kemungkinan imbalance
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f'\nscale_pos_weight (neg/pos) = {scale_pos_weight:.4f}')

### 3.3 Baseline Model XGBoost Binary

In [ ]:
xgb_base = XGBClassifier(
    objective='binary:logistic',   # ← binary classification
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight,  # ← tangani imbalance
    random_state=42,
    verbosity=0
)

xgb_base.fit(X_train, y_train)

y_pred_base      = xgb_base.predict(X_test)
y_prob_base      = xgb_base.predict_proba(X_test)[:, 1]

acc_base  = accuracy_score(y_test, y_pred_base)
prec_base = precision_score(y_test, y_pred_base)
rec_base  = recall_score(y_test, y_pred_base)
f1_base   = f1_score(y_test, y_pred_base)
auc_base  = roc_auc_score(y_test, y_prob_base)

print('=== Baseline XGBoost (Binary) – Test Set ===')
print(f'  Accuracy  : {acc_base:.4f}  ({acc_base*100:.2f}%)')
print(f'  Precision : {prec_base:.4f}')
print(f'  Recall    : {rec_base:.4f}')
print(f'  F1-Score  : {f1_base:.4f}')
print(f'  ROC-AUC   : {auc_base:.4f}')

### 3.4 Cross-Validation Baseline (5-Fold Stratified)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_acc  = cross_val_score(xgb_base, X_train, y_train, cv=cv, scoring='accuracy',  n_jobs=-1)
cv_f1   = cross_val_score(xgb_base, X_train, y_train, cv=cv, scoring='f1',         n_jobs=-1)
cv_auc  = cross_val_score(xgb_base, X_train, y_train, cv=cv, scoring='roc_auc',    n_jobs=-1)

print('=== 5-Fold Cross-Validation (Baseline) ===')
print(f'{'Fold':<8} {'Accuracy':>10} {'F1-Score':>10} {'ROC-AUC':>10}')
print('-' * 42)
for i in range(5):
    print(f'Fold {i+1:<3} {cv_acc[i]:>10.4f} {cv_f1[i]:>10.4f} {cv_auc[i]:>10.4f}')
print('-' * 42)
print(f'{'Mean':<8} {cv_acc.mean():>10.4f} {cv_f1.mean():>10.4f} {cv_auc.mean():>10.4f}')
print(f'{'Std':<8} {cv_acc.std():>10.4f} {cv_f1.std():>10.4f} {cv_auc.std():>10.4f}')

# Visualisasi CV
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(5)
width = 0.28
ax.bar(x - width, cv_acc,  width, label=f'Accuracy (μ={cv_acc.mean():.3f})',  color='#3498db', edgecolor='black')
ax.bar(x,         cv_f1,   width, label=f'F1-Score (μ={cv_f1.mean():.3f})',   color='#e74c3c', edgecolor='black')
ax.bar(x + width, cv_auc,  width, label=f'ROC-AUC  (μ={cv_auc.mean():.3f})',  color='#2ecc71', edgecolor='black')
ax.set_xlabel('Fold')
ax.set_ylabel('Score')
ax.set_title('5-Fold CV – Baseline XGBoost (Binary Classification)', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'Fold {i+1}' for i in range(5)])
ax.set_ylim(0.7, 1.02)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('03_cv_baseline_biner.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualisasi CV disimpan.')

### 3.5 Hyperparameter Tuning (RandomizedSearchCV)

In [ ]:
param_grid = {
    'n_estimators'     : [100, 200, 300, 400],
    'max_depth'        : [3, 4, 5, 6, 7],
    'learning_rate'    : [0.01, 0.05, 0.1, 0.2],
    'subsample'        : [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree' : [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight' : [1, 3, 5],
    'gamma'            : [0, 0.1, 0.2, 0.3],
    'reg_alpha'        : [0, 0.1, 0.5, 1.0],   # L1 regularization
    'reg_lambda'       : [0.5, 1.0, 1.5, 2.0]  # L2 regularization
}

xgb_tuning = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    verbosity=0
)

random_search = RandomizedSearchCV(
    estimator=xgb_tuning,
    param_distributions=param_grid,
    n_iter=50,
    scoring='f1',          # ← Optimasi F1 lebih tepat untuk deteksi kecanduan
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print('🔍 Memulai Hyperparameter Tuning (n_iter=50, scoring=F1)...')
random_search.fit(X_train, y_train)

print(f'\n✅ Best CV F1-Score : {random_search.best_score_:.4f}')
print(f'Best Parameters:')
for k, v in random_search.best_params_.items():
    print(f'  {k}: {v}')

### 3.6 Training Model Terbaik

In [ ]:
best_params = random_search.best_params_

xgb_best = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    verbosity=0,
    **best_params
)

eval_set = [(X_train, y_train), (X_test, y_test)]
xgb_best.fit(X_train, y_train, eval_set=eval_set, verbose=False)

y_pred_best = xgb_best.predict(X_test)
y_prob_best = xgb_best.predict_proba(X_test)[:, 1]

print('=== Perbandingan Baseline vs Best Model ===')
print(f'{'Metrik':<15} {'Baseline':>10} {'Best Model':>12} {'Delta':>10}')
print('-' * 50)
metrics = [
    ('Accuracy',  accuracy_score(y_test, y_pred_base),   accuracy_score(y_test, y_pred_best)),
    ('Precision', precision_score(y_test, y_pred_base),  precision_score(y_test, y_pred_best)),
    ('Recall',    recall_score(y_test, y_pred_base),     recall_score(y_test, y_pred_best)),
    ('F1-Score',  f1_score(y_test, y_pred_base),         f1_score(y_test, y_pred_best)),
    ('ROC-AUC',   roc_auc_score(y_test, y_prob_base),    roc_auc_score(y_test, y_prob_best)),
]
for name, base, best in metrics:
    delta = best - base
    sign  = '+' if delta >= 0 else ''
    print(f'{name:<15} {base:>10.4f} {best:>12.4f} {sign}{delta:>9.4f}')

### 3.7 Laporan Evaluasi Lengkap

In [ ]:
# Classification Report
print('=' * 60)
print('  LAPORAN EVALUASI – XGBoost Binary (Test Set)')
print('=' * 60)
print(classification_report(
    y_test, y_pred_best,
    target_names=['0 – Tidak Kecanduan', '1 – Kecanduan'],
    digits=4
))

# Ringkasan numerik
acc   = accuracy_score(y_test, y_pred_best)
prec  = precision_score(y_test, y_pred_best)
rec   = recall_score(y_test, y_pred_best)
f1    = f1_score(y_test, y_pred_best)
auc   = roc_auc_score(y_test, y_prob_best)
ap    = average_precision_score(y_test, y_prob_best)

print('=== Ringkasan Metrik ===')
print(f'  Accuracy          : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision         : {prec:.4f}')
print(f'  Recall (Sensitivity): {rec:.4f}')
print(f'  F1-Score          : {f1:.4f}')
print(f'  ROC-AUC           : {auc:.4f}')
print(f'  Avg Precision (PR): {ap:.4f}')
print(f'  Support (kecanduan)   : {(y_test==1).sum()}')
print(f'  Support (tdk kecanduan): {(y_test==0).sum()}')

In [ ]:
# === Confusion Matrix ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw count
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['0 – Tidak Kecanduan', '1 – Kecanduan'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix (Count)', fontweight='bold')

# Normalized
cm_norm = confusion_matrix(y_test, y_pred_best, normalize='true')
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_norm,
                                display_labels=['0 – Tidak Kecanduan', '1 – Kecanduan'])
disp2.plot(ax=axes[1], colorbar=False, cmap='Reds')
axes[1].set_title('Confusion Matrix (Normalized)', fontweight='bold')

tn, fp, fn, tp = cm.ravel()
print(f'TN={tn}  FP={fp}  FN={fn}  TP={tp}')
print(f'Specificity (True Negative Rate): {tn/(tn+fp):.4f}')
print(f'Sensitivity (Recall)            : {tp/(tp+fn):.4f}')

plt.suptitle('Confusion Matrix – XGBoost Binary Classifier', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('03_confusion_matrix_biner.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix disimpan.')

### 3.8 ROC Curve & Precision-Recall Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- ROC Curve ---
fpr, tpr, thresholds_roc = roc_curve(y_test, y_prob_best)
axes[0].plot(fpr, tpr, color='#e74c3c', lw=2,
             label=f'XGBoost (AUC = {auc:.4f})')
axes[0].plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random Classifier')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
axes[0].set_xlabel('False Positive Rate (1 - Specificity)')
axes[0].set_ylabel('True Positive Rate (Sensitivity / Recall)')
axes[0].set_title('ROC Curve – XGBoost Binary', fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(alpha=0.3)

# --- Precision-Recall Curve ---
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob_best)
axes[1].plot(recall_curve, precision_curve, color='#3498db', lw=2,
             label=f'XGBoost (AP = {ap:.4f})')
baseline_pr = (y_test == 1).sum() / len(y_test)
axes[1].axhline(baseline_pr, color='gray', linestyle='--', lw=1,
                label=f'Baseline (AP = {baseline_pr:.4f})')
axes[1].fill_between(recall_curve, precision_curve, alpha=0.1, color='#3498db')
axes[1].set_xlabel('Recall (Sensitivity)')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve – XGBoost Binary', fontweight='bold')
axes[1].legend(loc='upper right')
axes[1].grid(alpha=0.3)

plt.suptitle('Kurva Evaluasi Model – Deteksi Kecanduan Smartphone', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('03_roc_pr_curve_biner.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ ROC & PR Curve disimpan.')

### 3.9 Learning Curve (Training vs Test Loss)

In [ ]:
results = xgb_best.evals_result()
epochs  = len(results['validation_0']['logloss'])

plt.figure(figsize=(10, 5))
plt.plot(range(epochs), results['validation_0']['logloss'],
         label='Train Loss', color='#3498db', lw=2)
plt.plot(range(epochs), results['validation_1']['logloss'],
         label='Test Loss',  color='#e74c3c', lw=2, linestyle='--')
plt.xlabel('Epoch (n_estimators)')
plt.ylabel('Log Loss (Binary Cross-Entropy)')
plt.title('Learning Curve – XGBoost Binary Classification', fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('03_learning_curve_biner.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Learning curve disimpan.')

### 3.10 Feature Importance (XGBoost Built-in + Permutation)

In [ ]:
from sklearn.inspection import permutation_importance

# XGBoost built-in importance (gain)
importance_gain = pd.Series(
    xgb_best.get_booster().get_score(importance_type='gain'),
    name='Gain'
).sort_values(ascending=False)

# Permutation importance
perm_imp = permutation_importance(
    xgb_best, X_test, y_test,
    n_repeats=10, random_state=42, scoring='f1'
)
perm_df = pd.DataFrame({
    'Feature'        : X_test.columns,
    'Importance_Mean': perm_imp.importances_mean,
    'Importance_Std' : perm_imp.importances_std
}).sort_values('Importance_Mean', ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot Gain Importance
top_features = importance_gain.head(10)
axes[0].barh(top_features.index[::-1], top_features.values[::-1],
             color='#3498db', edgecolor='black')
axes[0].set_title('XGBoost Feature Importance (Gain)', fontweight='bold')
axes[0].set_xlabel('Gain Score')

# Plot Permutation Importance
axes[1].barh(perm_df['Feature'][::-1], perm_df['Importance_Mean'][::-1],
             xerr=perm_df['Importance_Std'][::-1],
             color='#e74c3c', edgecolor='black', capsize=4)
axes[1].set_title('Permutation Feature Importance (F1-Score Decrease)', fontweight='bold')
axes[1].set_xlabel('Mean Accuracy Decrease')

plt.suptitle('Analisis Feature Importance – Deteksi Kecanduan Smartphone', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('03_feature_importance_biner.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== Top Feature Importance (Permutation) ===')
print(perm_df.to_string(index=False))

### 3.11 Tabel Rekapitulasi Evaluasi Lengkap

In [ ]:
# Hitung semua metrik secara terperinci
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_best).ravel()

eval_table = {
    'Metrik'   : [
        'Accuracy', 'Precision', 'Recall (Sensitivity)',
        'Specificity', 'F1-Score', 'ROC-AUC',
        'Avg Precision (PR-AUC)',
        'True Positive (TP)', 'True Negative (TN)',
        'False Positive (FP)', 'False Negative (FN)',
        'Support (Kecanduan)', 'Support (Tidak Kecanduan)'
    ],
    'Nilai'    : [
        f'{acc:.4f}  ({acc*100:.2f}%)',
        f'{prec:.4f}',
        f'{rec:.4f}',
        f'{tn/(tn+fp):.4f}',
        f'{f1:.4f}',
        f'{auc:.4f}',
        f'{ap:.4f}',
        str(int(tp)), str(int(tn)),
        str(int(fp)), str(int(fn)),
        str(int((y_test==1).sum())),
        str(int((y_test==0).sum()))
    ],
    'Keterangan': [
        'Proporsi prediksi benar secara keseluruhan',
        'Dari yang diprediksi kecanduan, berapa % benar kecanduan',
        'Dari yang benar kecanduan, berapa % terdeteksi',
        'Dari yang benar tidak kecanduan, berapa % terdeteksi',
        'Harmonic mean Precision & Recall',
        'Area di bawah kurva ROC',
        'Area di bawah kurva Precision-Recall',
        'Kecanduan → prediksi kecanduan ✅',
        'Tidak kecanduan → prediksi tidak kecanduan ✅',
        'Tidak kecanduan → diprediksi kecanduan ❌ (False Alarm)',
        'Kecanduan → diprediksi tidak kecanduan ❌ (Missed)',
        'Jumlah sampel kecanduan di test set',
        'Jumlah sampel tidak kecanduan di test set'
    ]
}

eval_df = pd.DataFrame(eval_table)
print('=' * 90)
print('  TABEL EVALUASI LENGKAP – XGBoost Binary Classifier')
print('  Dataset: User Behavior | Target: Deteksi Kecanduan Smartphone')
print('=' * 90)
print(eval_df.to_string(index=False))

eval_df.to_csv('03_tabel_evaluasi_lengkap.csv', index=False)
print('\n✅ Tabel evaluasi disimpan ke: 03_tabel_evaluasi_lengkap.csv')

### 3.12 Simpan Model

In [ ]:
joblib.dump(xgb_best,    'xgboost_binary_model.pkl')
joblib.dump(best_params, 'best_params_binary.pkl')

print('✅ Model tersimpan:')
print('   - xgboost_binary_model.pkl')
print('   - best_params_binary.pkl')
print('\nParameter terbaik:')
for k, v in best_params.items():
    print(f'   {k}: {v}')

print(f'\n📊 Ringkasan Akhir Model:')
print(f'   Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'   Precision : {prec:.4f}')
print(f'   Recall    : {rec:.4f}')
print(f'   F1-Score  : {f1:.4f}')
print(f'   ROC-AUC   : {auc:.4f}')
print(f'   Output Label: 0 = Tidak Kecanduan | 1 = Kecanduan')